# 04 Analysis: Explainability with LIME and SHAP

Notebook 03 already performs model evaluation. It loads the trained models, runs inference for XLM-R and AfriBERTa on Hausa and Kinyarwanda, computes accuracy, precision, recall and F1-score, and saves the model disagreement CSV files.

This notebook focuses on explainability analysis. It loads the disagreement cases from notebook 03 and applies LIME and SHAP to selected examples so we can inspect which tokens influenced each model's prediction.

The main outputs from this notebook are:

- Selected disagreement examples for explainability
- LIME local explanation HTML files
- LIME token-weight CSV summaries
- SHAP token-attribution CSV summaries
- Case-level comparison tables for reporting

### How to set runtime:

Runtime > Change runtime type > Hardware accelerator > T4 GPU

## 1. Install Required Packages

LIME and SHAP are used for local model explainability. These packages are not always available by default in Colab, so we install them here.

In [1]:
!pip -q install lime shap transformers accelerate scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 14.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


## 2. Runtime Setup and Imports

This notebook should be run with a GPU runtime.

In Colab, use:

Runtime > Change runtime type > Hardware accelerator > T4 GPU

The notebook stops early if CUDA is unavailable, because LIME and SHAP repeatedly call the models and will be much slower on CPU.

In [2]:


import os
import re
import gc
import warnings

import numpy as np
import pandas as pd
import torch

from google.colab import drive
from IPython.display import display, HTML

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from lime.lime_text import LimeTextExplainer
import shap

warnings.filterwarnings("ignore")

# Mount Google Drive only if it is not already mounted
if not os.path.exists("/content/drive/MyDrive") and not os.path.exists("/content/drive/Shareddrives"):
    drive.mount("/content/drive")
else:
    print("Google Drive already appears to be mounted.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("CUDA available:", torch.cuda.is_available())
print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "CUDA is not available. Go to Runtime > Change runtime type and select T4 GPU before running notebook 04."
    )

Mounted at /content/drive
CUDA available: True
Using device: cuda
GPU: Tesla T4


## 3. Define Project Paths and Constants

The disagreement CSV files were created in notebook 03 and saved under `outputs/metrics`.

This notebook loads those disagreement files, then loads the trained models only when needed for LIME or SHAP explanations.

In [3]:
PROJECT_PATH = "/content/drive/Shareddrives/Cos760"

LANGUAGES = ["hausa", "kinyarwanda"]
MODELS = ["xlmr", "afriberta"]

CLASS_NAMES = ["negative", "neutral", "positive"]

LBL2ID = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

ID2LBL = {
    0: "negative",
    1: "neutral",
    2: "positive"
}

MODEL_PATHS = {
    "xlmr": {
        lang: os.path.join(PROJECT_PATH, "models", "xlmr", lang, "final")
        for lang in LANGUAGES
    },
    "afriberta": {
        lang: os.path.join(PROJECT_PATH, "models", "afriberta", lang, "final")
        for lang in LANGUAGES
    }
}

DISAGREEMENT_PATHS = {
    lang: os.path.join(PROJECT_PATH, "outputs", "metrics", f"disagreements_{lang}.csv")
    for lang in LANGUAGES
}

XAI_DIR = os.path.join(PROJECT_PATH, "outputs", "analysis", "xai")
LIME_DIR = os.path.join(XAI_DIR, "lime")
SHAP_DIR = os.path.join(XAI_DIR, "shap")
TABLE_DIR = os.path.join(XAI_DIR, "tables")

for path in [XAI_DIR, LIME_DIR, SHAP_DIR, TABLE_DIR]:
    os.makedirs(path, exist_ok=True)

print("Checking model paths:")
for model_name, lang_paths in MODEL_PATHS.items():
    for lang, path in lang_paths.items():
        print(f"{model_name}/{lang}: {path} | exists={os.path.exists(path)}")

print("\nChecking disagreement files:")
for lang, path in DISAGREEMENT_PATHS.items():
    print(f"{lang}: {path} | exists={os.path.exists(path)}")

Checking model paths:
xlmr/hausa: /content/drive/Shareddrives/Cos760/models/xlmr/hausa/final | exists=True
xlmr/kinyarwanda: /content/drive/Shareddrives/Cos760/models/xlmr/kinyarwanda/final | exists=True
afriberta/hausa: /content/drive/Shareddrives/Cos760/models/afriberta/hausa/final | exists=True
afriberta/kinyarwanda: /content/drive/Shareddrives/Cos760/models/afriberta/kinyarwanda/final | exists=True

Checking disagreement files:
hausa: /content/drive/Shareddrives/Cos760/outputs/metrics/disagreements_hausa.csv | exists=True
kinyarwanda: /content/drive/Shareddrives/Cos760/outputs/metrics/disagreements_kinyarwanda.csv | exists=True


In [4]:



import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    accuracy_score, confusion_matrix, classification_report
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


PROJECT_PATH = '/content/drive/Shareddrives/Cos760'

# label mappings
LBL2ID = {'negative': 0, 'neutral': 1, 'positive': 2}
ID2LBL = {0: 'negative', 1: 'neutral', 2: 'positive'}
LANGUAGES = ['hausa', 'kinyarwanda']

# model paths
MODEL_PATHS = {
    'xlmr': {
        lang: os.path.join(PROJECT_PATH, f'models/xlmr/{lang}/final')
        for lang in LANGUAGES
    },
    'afriberta': {
        lang: os.path.join(PROJECT_PATH, f'models/afriberta/{lang}/final')
        for lang in LANGUAGES
    }
}

print("Setup complete. Model paths:")
for model_name, langs in MODEL_PATHS.items():
    for lang, path in langs.items():
        exists = os.path.exists(path)
        print(f"  [{('OK' if exists else 'MISSING')}] {model_name}/{lang}: {path}")

Using device: cuda
Setup complete. Model paths:
  [OK] xlmr/hausa: /content/drive/Shareddrives/Cos760/models/xlmr/hausa/final
  [OK] xlmr/kinyarwanda: /content/drive/Shareddrives/Cos760/models/xlmr/kinyarwanda/final
  [OK] afriberta/hausa: /content/drive/Shareddrives/Cos760/models/afriberta/hausa/final
  [OK] afriberta/kinyarwanda: /content/drive/Shareddrives/Cos760/models/afriberta/kinyarwanda/final


## 4. Load Disagreement Files

The saved disagreement files contain only the tweets where XLM-R and AfriBERTa predicted different labels.

For each disagreement case, we add:

- the language,
- a unique case ID,
- the cleaned text to analyse,
- whether each model was correct,
- and a case type describing the disagreement outcome.

In [5]:
def normalise_label(value):
    """
    Converts labels into the consistent string format:
    negative, neutral, positive.
    """

    if pd.isna(value):
        return value

    value = str(value).strip().lower()

    label_map = {
        "0": "negative",
        "1": "neutral",
        "2": "positive",
        "label_0": "negative",
        "label_1": "neutral",
        "label_2": "positive",
        "negative": "negative",
        "neutral": "neutral",
        "positive": "positive"
    }

    return label_map.get(value, value)


def find_text_column(df):
    """
    Finds the most likely text column in the disagreement CSV.
    Notebook 03 uses cleaned_tweet, but this function makes the notebook safer.
    """

    candidates = [
        "cleaned_tweet",
        "tweet",
        "text",
        "sentence",
        "content",
        "clean_text",
        "processed_text"
    ]

    for col in candidates:
        if col in df.columns:
            return col

    object_cols = df.select_dtypes(include=["object"]).columns.tolist()

    if object_cols:
        return object_cols[0]

    raise ValueError("Could not identify a text column in the disagreement file.")


def load_disagreement_file(lang):
    path = DISAGREEMENT_PATHS[lang]

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Missing disagreement file: {path}. Run notebook 03 first."
        )

    df = pd.read_csv(path)
    text_col = find_text_column(df)

    required_columns = ["true_label", "xlmr_pred", "afriberta_pred"]
    missing_columns = [col for col in required_columns if col not in df.columns]

    if missing_columns:
        raise ValueError(
            f"{path} is missing required columns: {missing_columns}"
        )

    df = df.copy()

    df["language"] = lang
    df["case_id"] = [f"{lang}_{i}" for i in range(len(df))]
    df["analysis_text"] = df[text_col].astype(str)

    df["true_label"] = df["true_label"].apply(normalise_label)
    df["xlmr_pred"] = df["xlmr_pred"].apply(normalise_label)
    df["afriberta_pred"] = df["afriberta_pred"].apply(normalise_label)

    df["xlmr_correct"] = df["xlmr_pred"] == df["true_label"]
    df["afriberta_correct"] = df["afriberta_pred"] == df["true_label"]

    conditions = [
        (df["xlmr_correct"] == True) & (df["afriberta_correct"] == False),
        (df["xlmr_correct"] == False) & (df["afriberta_correct"] == True),
        (df["xlmr_correct"] == False) & (df["afriberta_correct"] == False)
    ]

    choices = [
        "xlmr_correct_afriberta_wrong",
        "afriberta_correct_xlmr_wrong",
        "both_wrong"
    ]

    df["case_type"] = np.select(
        conditions,
        choices,
        default="other"
    )

    df["word_count"] = df["analysis_text"].str.split().str.len()

    return df, text_col


disagreements = {}
text_columns = {}

for lang in LANGUAGES:
    df, text_col = load_disagreement_file(lang)

    disagreements[lang] = df
    text_columns[lang] = text_col

    print("\n" + "=" * 80)
    print(lang.upper())
    print("=" * 80)
    print("Rows:", len(df))
    print("Detected text column:", text_col)
    print("\nCase type counts:")
    print(df["case_type"].value_counts().to_string())

disagreement_all_df = pd.concat(disagreements.values(), ignore_index=True)

display(
    disagreement_all_df[
        [
            "case_id",
            "language",
            "true_label",
            "xlmr_pred",
            "afriberta_pred",
            "xlmr_correct",
            "afriberta_correct",
            "case_type",
            "analysis_text"
        ]
    ].head()
)


HAUSA
Rows: 1192
Detected text column: cleaned_tweet

Case type counts:
case_type
afriberta_correct_xlmr_wrong    695
xlmr_correct_afriberta_wrong    398
both_wrong                       99

KINYARWANDA
Rows: 400
Detected text column: cleaned_tweet

Case type counts:
case_type
afriberta_correct_xlmr_wrong    208
xlmr_correct_afriberta_wrong    126
both_wrong                       66


,case_id,language,true_label,xlmr_pred,afriberta_pred,xlmr_correct,afriberta_correct,case_type,analysis_text
0,hausa_0,hausa,positive,negative,neutral,False,False,both_wrong,wannan shine gaskiyar lamari domin siyasar nig...
1,hausa_1,hausa,positive,negative,positive,False,True,afriberta_correct_xlmr_wrong,kaeee dis one is too much burgewa wallahi
2,hausa_2,hausa,positive,negative,neutral,False,False,both_wrong,buqatan maji haji sallah
3,hausa_3,hausa,positive,neutral,positive,False,True,afriberta_correct_xlmr_wrong,zango zango sannu da kokari
4,hausa_4,hausa,positive,neutral,positive,False,True,afriberta_correct_xlmr_wrong,lolz duk inda ta kwana sha ne thts very right


## 5. Summarise Disagreement Outcomes

Before applying LIME and SHAP, we summarise how often each type of disagreement occurs.

This helps us choose representative cases for explainability rather than explaining every disagreement case.

In [6]:
disagreement_summary = (
    disagreement_all_df
    .groupby(["language", "case_type"])
    .size()
    .reset_index(name="count")
    .sort_values(["language", "case_type"])
)

summary_output_path = os.path.join(TABLE_DIR, "disagreement_case_type_summary.csv")
disagreement_summary.to_csv(summary_output_path, index=False)

print("Saved disagreement summary to:", summary_output_path)
display(disagreement_summary)

Saved disagreement summary to: /content/drive/Shareddrives/Cos760/outputs/analysis/xai/tables/disagreement_case_type_summary.csv


,language,case_type,count
0,hausa,afriberta_correct_xlmr_wrong,695
1,hausa,both_wrong,99
2,hausa,xlmr_correct_afriberta_wrong,398
3,kinyarwanda,afriberta_correct_xlmr_wrong,208
4,kinyarwanda,both_wrong,66
5,kinyarwanda,xlmr_correct_afriberta_wrong,126


## 6. Select Representative Cases for XAI

LIME and SHAP are local explanation methods. They explain individual predictions, not the entire dataset at once.

To keep the notebook practical, we select a small number of representative disagreement cases from each language and disagreement type.

The default is one example per case type per language.

In [7]:
RANDOM_SEED = 42
N_PER_BUCKET = 1

CASE_TYPES = [
    "xlmr_correct_afriberta_wrong",
    "afriberta_correct_xlmr_wrong",
    "both_wrong"
]

selected_parts = []

for lang in LANGUAGES:
    lang_df = disagreements[lang]

    for case_type in CASE_TYPES:
        subset = lang_df[lang_df["case_type"] == case_type]

        if len(subset) == 0:
            print(f"No cases found for {lang} / {case_type}")
            continue

        selected_parts.append(
            subset.sample(
                n=min(N_PER_BUCKET, len(subset)),
                random_state=RANDOM_SEED
            )
        )

selected_cases_df = pd.concat(selected_parts, ignore_index=True)

selected_cases_path = os.path.join(TABLE_DIR, "selected_xai_cases.csv")
selected_cases_df.to_csv(selected_cases_path, index=False)

print("Selected XAI cases:", len(selected_cases_df))
print("Saved selected cases to:", selected_cases_path)

display(
    selected_cases_df[
        [
            "case_id",
            "language",
            "case_type",
            "true_label",
            "xlmr_pred",
            "afriberta_pred",
            "analysis_text"
        ]
    ]
)

Selected XAI cases: 6
Saved selected cases to: /content/drive/Shareddrives/Cos760/outputs/analysis/xai/tables/selected_xai_cases.csv


,case_id,language,case_type,true_label,xlmr_pred,afriberta_pred,analysis_text
0,hausa_642,hausa,xlmr_correct_afriberta_wrong,neutral,neutral,negative,buhari kuma ya doke atiku a nigeria
1,hausa_647,hausa,afriberta_correct_xlmr_wrong,neutral,negative,neutral,gaskiya kunsa naje hira wajan budurwata kachem...
2,hausa_607,hausa,both_wrong,neutral,negative,positive,saboda qarancin oxygen da ake samu idan aka sa...
3,kinyarwanda_230,kinyarwanda,xlmr_correct_afriberta_wrong,neutral,neutral,positive,3/3 6️⃣ Gukomeza kurebera hamwe uko Amabwiriza...
4,kinyarwanda_310,kinyarwanda,afriberta_correct_xlmr_wrong,neutral,positive,neutral,Ikintu cya mbere ni amakuru. Abanyamakuru mwes...
5,kinyarwanda_350,kinyarwanda,both_wrong,neutral,negative,positive,Munyemerere Tuganire kandi twungurane ubumenyi...


## 7. Define Prediction Wrapper for LIME and SHAP

LIME and SHAP both need a prediction function that accepts a list of texts and returns class probabilities.

The wrapper below loads one model at a time and exposes a `predict_proba()` method. This prevents the notebook from keeping all four models in memory at once.

In [8]:
BATCH_SIZE = 32
MAX_LENGTH = 128


def clear_memory():
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


class TransformerSentimentPredictor:
    """
    Wrapper around a Hugging Face sequence classification model.

    LIME and SHAP call predict_proba repeatedly using perturbed versions of the
    original text. This wrapper batches those calls and returns probabilities.
    """

    def __init__(
        self,
        model_path,
        device=device,
        batch_size=BATCH_SIZE,
        max_length=MAX_LENGTH
    ):
        self.model_path = model_path
        self.device = device
        self.batch_size = batch_size
        self.max_length = max_length

        self.tokenizer = AutoTokenizer.from_pretrained(
            model_path,
            local_files_only=True
        )

        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_path,
            local_files_only=True
        )

        self.model.to(device)
        self.model.eval()

    def predict_proba(self, texts):
        if isinstance(texts, str):
            texts = [texts]

        texts = [str(text) for text in texts]
        all_probs = []

        with torch.no_grad():
            for start in range(0, len(texts), self.batch_size):
                batch_texts = texts[start:start + self.batch_size]

                encoded = self.tokenizer(
                    batch_texts,
                    padding=True,
                    truncation=True,
                    max_length=self.max_length,
                    return_tensors="pt"
                )

                encoded = {
                    key: value.to(self.device)
                    for key, value in encoded.items()
                }

                outputs = self.model(**encoded)
                probs = torch.softmax(outputs.logits, dim=-1)

                all_probs.append(probs.detach().cpu().numpy())

        return np.vstack(all_probs)

    def close(self):
        del self.model
        del self.tokenizer
        clear_memory()


def safe_filename(text):
    text = str(text)
    text = re.sub(r"[^A-Za-z0-9_.-]+", "_", text)
    return text[:120]


def model_prediction_columns(model_name):
    if model_name == "xlmr":
        return "xlmr_pred", "xlmr_correct"

    if model_name == "afriberta":
        return "afriberta_pred", "afriberta_correct"

    raise ValueError("model_name must be either 'xlmr' or 'afriberta'.")

## 8. Run LIME Explanations

LIME explains a prediction by perturbing the input text and learning which words most influenced the model's output.

For each selected disagreement case, we explain the class predicted by the model. The outputs are saved as:

- HTML explanation files
- a CSV containing the ranked token weights

In [ ]:
LIME_NUM_FEATURES = 12
LIME_NUM_SAMPLES = 1000


def run_lime_for_cases(model_name, lang, cases_df):
    model_path = MODEL_PATHS[model_name][lang]

    print("\n" + "=" * 80)
    print(f"Running LIME for {model_name.upper()} on {lang.upper()}")
    print("=" * 80)

    predictor = TransformerSentimentPredictor(model_path)

    explainer = LimeTextExplainer(
        class_names=CLASS_NAMES,
        random_state=RANDOM_SEED
    )

    rows = []

    for _, row in cases_df.iterrows():
        text = row["analysis_text"]

        pred_col, correct_col = model_prediction_columns(model_name)

        predicted_label = row[pred_col]
        target_id = LBL2ID[predicted_label]

        print(
            f"LIME | model={model_name} | lang={lang} | case={row['case_id']} | predicted={predicted_label}"
        )

        explanation = explainer.explain_instance(
            text_instance=text,
            classifier_fn=predictor.predict_proba,
            labels=[target_id],
            num_features=LIME_NUM_FEATURES,
            num_samples=LIME_NUM_SAMPLES
        )

        html_name = safe_filename(
            f"lime_{lang}_{model_name}_{row['case_id']}_{predicted_label}.html"
        )

        html_path = os.path.join(LIME_DIR, html_name)
        explanation.save_to_file(html_path)

        for rank, feature_weight in enumerate(
            explanation.as_list(label=target_id),
            start=1
        ):
            feature, weight = feature_weight

            rows.append({
                "method": "lime",
                "language": lang,
                "model": model_name,
                "case_id": row["case_id"],
                "case_type": row["case_type"],
                "true_label": row["true_label"],
                "predicted_label": predicted_label,
                "model_correct": bool(row[correct_col]),
                "rank": rank,
                "feature": feature,
                "weight": float(weight),
                "abs_weight": abs(float(weight)),
                "html_path": html_path,
                "text": text
            })

    predictor.close()

    return rows


lime_rows = []

for lang in LANGUAGES:
    cases_lang = selected_cases_df[selected_cases_df["language"] == lang]

    for model_name in MODELS:
        lime_rows.extend(
            run_lime_for_cases(model_name, lang, cases_lang)
        )

lime_df = pd.DataFrame(lime_rows)

lime_output_path = os.path.join(TABLE_DIR, "lime_token_weights.csv")
lime_df.to_csv(lime_output_path, index=False)

print("\nSaved LIME token weights to:", lime_output_path)

display(lime_df.head(20))


Running LIME for XLMR on HAUSA


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

LIME | model=xlmr | lang=hausa | case=hausa_642 | predicted=neutral
LIME | model=xlmr | lang=hausa | case=hausa_647 | predicted=negative
LIME | model=xlmr | lang=hausa | case=hausa_607 | predicted=negative

Running LIME for AFRIBERTA on HAUSA


Loading weights:   0%|          | 0/169 [00:00<?, ?it/s]

## 9. Run SHAP Explanations

SHAP estimates how much each token contributed to the model's predicted class.

SHAP can be slower than LIME, so this notebook only applies SHAP to the selected representative disagreement cases.

The outputs are saved as:

- SHAP token attribution CSV files
- SHAP HTML visualisations where possible

In [ ]:
SHAP_MAX_EVALS = 300
SHAP_NUM_FEATURES = 12


def extract_shap_token_values(shap_values, target_id):
    """
    Extracts tokens and SHAP values for the predicted class.

    SHAP output shapes can vary slightly by version, so this function handles
    the common text-classification layouts.
    """

    tokens = np.array(shap_values.data[0]).astype(str)
    values = np.array(shap_values.values[0])

    if values.ndim == 2:
        if values.shape[1] == len(CLASS_NAMES):
            target_values = values[:, target_id]
        elif values.shape[0] == len(CLASS_NAMES):
            target_values = values[target_id, :]
        else:
            raise ValueError(f"Unexpected SHAP value shape: {values.shape}")
    elif values.ndim == 1:
        target_values = values
    else:
        raise ValueError(f"Unexpected SHAP value dimensions: {values.ndim}")

    usable_length = min(len(tokens), len(target_values))

    tokens = tokens[:usable_length]
    target_values = target_values[:usable_length]

    return tokens, target_values


def save_shap_html(shap_values, target_id, output_path):
    """
    Attempts to save SHAP's text plot as HTML.
    If the installed SHAP version does not return HTML cleanly, the CSV output
    is still saved and used for reporting.
    """

    try:
        shap_text = shap_values[0, :, target_id]
        html_obj = shap.plots.text(shap_text, display=False)

        html_content = getattr(html_obj, "data", str(html_obj))

        with open(output_path, "w", encoding="utf-8") as f:
            f.write(html_content)

        return output_path

    except Exception as exc:
        print(f"Could not save SHAP HTML: {exc}")
        return ""


def run_shap_for_cases(model_name, lang, cases_df):
    model_path = MODEL_PATHS[model_name][lang]

    print("\n" + "=" * 80)
    print(f"Running SHAP for {model_name.upper()} on {lang.upper()}")
    print("=" * 80)

    predictor = TransformerSentimentPredictor(model_path)

    masker = shap.maskers.Text(predictor.tokenizer)

    explainer = shap.Explainer(
        predictor.predict_proba,
        masker,
        output_names=CLASS_NAMES
    )

    rows = []
    error_rows = []

    for _, row in cases_df.iterrows():
        text = row["analysis_text"]

        pred_col, correct_col = model_prediction_columns(model_name)

        predicted_label = row[pred_col]
        target_id = LBL2ID[predicted_label]

        print(
            f"SHAP | model={model_name} | lang={lang} | case={row['case_id']} | predicted={predicted_label}"
        )

        try:
            shap_values = explainer(
                [text],
                max_evals=SHAP_MAX_EVALS,
                batch_size=BATCH_SIZE
            )

            tokens, values = extract_shap_token_values(
                shap_values=shap_values,
                target_id=target_id
            )

            html_name = safe_filename(
                f"shap_{lang}_{model_name}_{row['case_id']}_{predicted_label}.html"
            )

            html_path = os.path.join(SHAP_DIR, html_name)

            html_path = save_shap_html(
                shap_values=shap_values,
                target_id=target_id,
                output_path=html_path
            )

            order = np.argsort(np.abs(values))[::-1][:SHAP_NUM_FEATURES]

            for rank, idx in enumerate(order, start=1):
                rows.append({
                    "method": "shap",
                    "language": lang,
                    "model": model_name,
                    "case_id": row["case_id"],
                    "case_type": row["case_type"],
                    "true_label": row["true_label"],
                    "predicted_label": predicted_label,
                    "model_correct": bool(row[correct_col]),
                    "rank": rank,
                    "feature": tokens[idx],
                    "weight": float(values[idx]),
                    "abs_weight": abs(float(values[idx])),
                    "html_path": html_path,
                    "text": text
                })

        except Exception as exc:
            print(f"SHAP failed for {row['case_id']}: {exc}")

            error_rows.append({
                "language": lang,
                "model": model_name,
                "case_id": row["case_id"],
                "error": str(exc),
                "text": text
            })

    predictor.close()

    return rows, error_rows


shap_rows = []
shap_error_rows = []

for lang in LANGUAGES:
    cases_lang = selected_cases_df[selected_cases_df["language"] == lang]

    for model_name in MODELS:
        rows, errors = run_shap_for_cases(model_name, lang, cases_lang)
        shap_rows.extend(rows)
        shap_error_rows.extend(errors)

shap_df = pd.DataFrame(shap_rows)

shap_output_path = os.path.join(TABLE_DIR, "shap_token_attributions.csv")
shap_df.to_csv(shap_output_path, index=False)

print("\nSaved SHAP token attributions to:", shap_output_path)

if len(shap_error_rows) > 0:
    shap_errors_df = pd.DataFrame(shap_error_rows)
    shap_errors_path = os.path.join(TABLE_DIR, "shap_errors.csv")
    shap_errors_df.to_csv(shap_errors_path, index=False)
    print("Saved SHAP errors to:", shap_errors_path)

display(shap_df.head(20))

## 10. Summarise Important Features

The token-level LIME and SHAP outputs are aggregated into compact summary tables.

These tables help identify which words or tokens had the strongest influence on model predictions across the selected cases.

In [ ]:
def summarise_top_features(explanation_df, method_name, top_n=15):
    if explanation_df.empty:
        print(f"No rows available for {method_name}.")
        return pd.DataFrame()

    temp = explanation_df.copy()
    temp["abs_weight"] = temp["weight"].abs()

    summary = (
        temp
        .groupby(["language", "model", "feature"], as_index=False)
        .agg(
            mean_abs_weight=("abs_weight", "mean"),
            mean_weight=("weight", "mean"),
            frequency=("feature", "size")
        )
        .sort_values(
            ["language", "model", "mean_abs_weight"],
            ascending=[True, True, False]
        )
    )

    top_summary = (
        summary
        .groupby(["language", "model"], group_keys=False)
        .head(top_n)
        .reset_index(drop=True)
    )

    output_path = os.path.join(
        TABLE_DIR,
        f"{method_name}_top_features_summary.csv"
    )

    top_summary.to_csv(output_path, index=False)

    print(f"Saved {method_name.upper()} top feature summary to:", output_path)

    return top_summary


lime_top_features = summarise_top_features(lime_df, "lime")
shap_top_features = summarise_top_features(shap_df, "shap")

print("\nTop LIME features")
display(lime_top_features)

print("\nTop SHAP features")
display(shap_top_features)

## 11. Summarise XAI Outputs and Generate Reporting Files

This section creates:

- top LIME and SHAP feature summaries,
- a case-level XAI table,
- a LIME vs SHAP comparison table,
- reporting notes for notebook 05,
- and a final list of generated files.

In [ ]:
def summarise_top_features(explanation_df, method_name, top_n=15):
    if explanation_df.empty:
        print(f"No rows available for {method_name}.")
        return pd.DataFrame()

    temp = explanation_df.copy()
    temp["abs_weight"] = temp["weight"].abs()

    summary = (
        temp
        .groupby(["language", "model", "feature"], as_index=False)
        .agg(
            mean_abs_weight=("abs_weight", "mean"),
            mean_weight=("weight", "mean"),
            frequency=("feature", "size")
        )
        .sort_values(
            ["language", "model", "mean_abs_weight"],
            ascending=[True, True, False]
        )
    )

    top_summary = (
        summary
        .groupby(["language", "model"], group_keys=False)
        .head(top_n)
        .reset_index(drop=True)
    )

    output_path = os.path.join(
        TABLE_DIR,
        f"{method_name}_top_features_summary.csv"
    )

    top_summary.to_csv(output_path, index=False)
    print(f"Saved {method_name.upper()} top features to: {output_path}")

    return top_summary


def collapse_features(df, method_name, top_n=5):
    if df.empty:
        return pd.DataFrame(
            columns=["case_id", "model", f"{method_name}_top_features"]
        )

    temp = df.copy()
    temp["abs_weight"] = temp["weight"].abs()

    temp = temp.sort_values(
        ["case_id", "model", "abs_weight"],
        ascending=[True, True, False]
    )

    rows = []

    for (case_id, model), group in temp.groupby(["case_id", "model"]):
        features = [
            f"{r.feature} ({r.weight:.3f})"
            for r in group.head(top_n).itertuples()
        ]

        rows.append({
            "case_id": case_id,
            "model": model,
            f"{method_name}_top_features": "; ".join(features)
        })

    return pd.DataFrame(rows)


# 1. Summarise important LIME and SHAP features
lime_top_features = summarise_top_features(lime_df, "lime")
shap_top_features = summarise_top_features(shap_df, "shap")

lime_case_features = collapse_features(lime_df, "lime", top_n=5)
shap_case_features = collapse_features(shap_df, "shap", top_n=5)

# 2. Create case-level XAI table
case_summary_rows = []

for _, row in selected_cases_df.iterrows():
    for model_name in MODELS:
        pred_col, correct_col = model_prediction_columns(model_name)

        case_summary_rows.append({
            "case_id": row["case_id"],
            "language": row["language"],
            "case_type": row["case_type"],
            "model": model_name,
            "true_label": row["true_label"],
            "predicted_label": row[pred_col],
            "model_correct": bool(row[correct_col]),
            "text": row["analysis_text"]
        })

case_summary_df = pd.DataFrame(case_summary_rows)

case_summary_df = case_summary_df.merge(
    lime_case_features,
    on=["case_id", "model"],
    how="left"
)

case_summary_df = case_summary_df.merge(
    shap_case_features,
    on=["case_id", "model"],
    how="left"
)

case_summary_path = os.path.join(TABLE_DIR, "xai_case_level_summary.csv")
case_summary_df.to_csv(case_summary_path, index=False)

print(f"Saved case-level XAI summary to: {case_summary_path}")

# 3. Compare LIME and SHAP outputs
lime_compare = collapse_features(lime_df, "lime", top_n=8)
shap_compare = collapse_features(shap_df, "shap", top_n=8)

lime_shap_comparison_df = lime_compare.merge(
    shap_compare,
    on=["case_id", "model"],
    how="outer"
)

metadata_cols = [
    "case_id",
    "language",
    "case_type",
    "model",
    "true_label",
    "predicted_label",
    "model_correct",
    "text"
]

case_metadata = case_summary_df[metadata_cols].drop_duplicates()

lime_shap_comparison_df = case_metadata.merge(
    lime_shap_comparison_df,
    on=["case_id", "model"],
    how="left"
)

lime_shap_comparison_path = os.path.join(
    TABLE_DIR,
    "lime_shap_comparison_by_case.csv"
)

lime_shap_comparison_df.to_csv(lime_shap_comparison_path, index=False)

print(f"Saved LIME vs SHAP comparison to: {lime_shap_comparison_path}")

# 4. Generate reporting notes
report_notes = []

report_notes.append("04 Analysis: Explainability Summary")
report_notes.append("=" * 60)
report_notes.append("")
report_notes.append("Purpose:")
report_notes.append(
    "Notebook 04 uses the disagreement cases from notebook 03 and applies LIME and SHAP to selected examples."
)
report_notes.append("")
report_notes.append("Selected XAI cases:")
report_notes.append(f"- Total selected cases: {len(selected_cases_df)}")
report_notes.append(f"- Selection rule: {N_PER_BUCKET} case(s) per language and disagreement type")
report_notes.append("")
report_notes.append("Generated outputs:")
report_notes.append(f"- Selected cases: {selected_cases_path}")
report_notes.append(f"- Disagreement summary: {summary_output_path}")
report_notes.append(f"- LIME token weights: {lime_output_path}")
report_notes.append(f"- SHAP token attributions: {shap_output_path}")
report_notes.append(f"- Case-level XAI summary: {case_summary_path}")
report_notes.append(f"- LIME vs SHAP comparison: {lime_shap_comparison_path}")

report_notes_text = "\n".join(report_notes)

report_notes_path = os.path.join(TABLE_DIR, "xai_reporting_notes.txt")

with open(report_notes_path, "w", encoding="utf-8") as f:
    f.write(report_notes_text)

print(f"Saved reporting notes to: {report_notes_path}")

# 5. List generated files
print("\nGenerated XAI files:")

for root, dirs, files in os.walk(XAI_DIR):
    for file in files:
        print(os.path.join(root, file))

print("\nCase-level XAI summary:")
display(case_summary_df)

print("\nLIME vs SHAP comparison:")
display(lime_shap_comparison_df)

print("\nReporting notes:")
print(report_notes_text)